In [1]:
import json, warnings
warnings.filterwarnings('ignore')
import pertpy as pt
import pandas as pd
import numpy as np
import scanpy as sc
import tqdm

import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(5, 5)
sc.settings.verbosity = 3
sc.logging.print_header()
sc.set_figure_params(dpi=100)

%load_ext autoreload
%autoreload 2 

## load raw data and filter genes not in prior knowledge

In [2]:
radata_k562 = sc.read_h5ad('ReplogleWeissman2022_K562_essential_filtered.h5ad')
radata_rpe1 = sc.read_h5ad('ReplogleWeissman2022_RPE1_essential_filtered.h5ad')


In [3]:
padata_dict = {
    'k562': radata_k562,
    'rpe1': radata_rpe1,
}

for ct, pad in padata_dict.items():
    pad.obs['perturbation'] = pad.obs['perturbation'].map(lambda x: 'control' if x.startswith('non-targeting') else x)

## preprocessing

In [5]:
processed_padata_dict = {}
for ct, pad_ct in padata_dict.items():
    print(f"Processing {ct}")
    pad = pad_ct.copy()
    pad.X = pad.X.astype(np.float32)
    sc.pp.filter_cells(pad, min_genes=200)
    sc.pp.filter_genes(pad, min_cells=50)
    num_cell_per_pert = pad.obs.groupby('perturbation').size()
    included_perts = num_cell_per_pert[lambda x: x > 50].index
    pad.layers["raw"] = pad.X.copy()
    sc.pp.normalize_total(pad, target_sum=1e4)
    sc.pp.log1p(pad)
    pad = pad[pad.obs['perturbation'].isin(included_perts)]
    ms = pt.tl.Mixscape()
    ms.perturbation_signature(
        pad,
        pert_key="perturbation",
        control="control",
    )
    ms.mixscape(
        pad,
        labels="perturbation",
        control="control",
        layer='X_pert'
    )
    print(f"filter {(pad.obs['mixscape_class_global']=='NP').sum()} NP cells")
    pad = pad[pad.obs['mixscape_class_global'] != 'NP']
    processed_padata_dict[ct] = pad




Processing k562


filter 121737 NP cells
Processing rpe1


filter 72420 NP cells


In [ ]:
pertgenes_sel = np.intersect1d(*[
    pad.obs["perturbation"].value_counts()[lambda x: x > 50].index
    for ct, pad in processed_padata_dict.items()
])

padata = sc.concat([pad[pad.obs['perturbation'].isin(pertgenes_sel)] for ct, pad in processed_padata_dict.items()])
sc.pp.highly_variable_genes(padata, n_top_genes=5000)

padata.write_h5ad("preprocessed.h5ad")

## keep strong perts


In [ ]:
padata = sc.read("preprocessed.h5ad")

In [7]:
ptgs_csp = np.intersect1d(*[
    padata.obs.loc[(padata.obs['cell_type']==ct) & padata.obs['strong_perts'], 'perturbation'].unique()
    for ct in padata.obs['cell_type'].unique()
])
print(f"Selected {len(ptgs_csp)} perturbations")

Selected 362 perturbations


In [10]:
padata = padata[padata.obs['perturbation'].isin(ptgs_csp)]

In [ ]:
padata.write_h5ad("preprocessed_strong.h5ad")

## split dataset

In [ ]:
adata = sc.read(f'preprocessed.h5ad', backed='r')

In [ ]:
np.random.seed(42)
split_df = (
    adata
    .obs[['cell_type']]
    .copy()
    .reset_index(names='cell')
    .rename(columns={'cell_type': 'subsplit'})
)
split_df['presplit'] = np.random.choice(
    ['train', 'val', 'test'], 
    size=split_df.shape[0], 
    p=[0.7, 0.1, 0.2], 
    replace=True
)


In [ ]:
# train on K562 and test on RPE1
split_df['split'] = split_df[['subsplit', 'presplit']].apply(
    lambda x: x['presplit'] if x['subsplit'] == 'K562' else 'test', axis=1
)
split_df[['cell', 'split', 'subsplit']].to_csv('split_trainonk562_50.csv', index=False)
pd.read_csv('split_trainonk562.csv').groupby(['split', 'subsplit']).size()

split  subsplit
test   K562         34890
       RPE1        156748
train  K562        121854
val    K562         17422
dtype: int64

In [ ]:
# train on RPE1 and test on K562
split_df['split'] = split_df[['subsplit', 'presplit']].apply(
    lambda x: x['presplit'] if x['subsplit'] == 'RPE1' else 'test', axis=1
)
split_df[['cell', 'split', 'subsplit']].to_csv('split_trainonrpe1_50.csv', index=False)
pd.read_csv('split_trainonrpe1.csv').groupby(['split', 'subsplit']).size()

split  subsplit
test   K562        174166
       RPE1         31376
train  RPE1        109704
val    RPE1         15668
dtype: int64